In [ ]:
CREATE OR REPLACE PROCEDURE PROC_CREATE_SILVER(
    DB STRING,
    SCHEMA STRING,
    TABLE_MAPPINGS ARRAY
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sql_expr, lit
from functools import reduce
import operator

def main(session: Session, DB: str, SCHEMA: str, TABLE_MAPPINGS: list):

    now = sql_expr("TO_TIMESTAMP_NTZ(TO_CHAR(CONVERT_TIMEZONE('UTC', CURRENT_TIMESTAMP()), 'YYYY-MM-DD HH24:MI:SS'))")

    successes = []
    failures = []

    for mapping in TABLE_MAPPINGS:
        try:
            bronze_table = mapping['bronze_table']
            silver_table = mapping['silver_table']
            bronze_full = f"{DB}.{SCHEMA}.{bronze_table}"
            silver_full = f"{DB}.{SCHEMA}.{silver_table}"

            # ---------------------------------------
            # 1. Load bronze data
            # ---------------------------------------
            df = session.table(bronze_full)
            cols = df.columns
            cols_upper = [c.upper() for c in cols]

            # ---------------------------------------
            # 2. Dynamic column-type cleaning
            # ---------------------------------------
            schema_info = df.schema.fields

            for field in schema_info:
                cname = field.name
                ctype = str(field.datatype).upper()

                # Trim string columns
                if "STRING" in ctype or "TEXT" in ctype:
                    df = df.with_column(cname, sql_expr(f"TRIM({cname})"))

                # Validate numeric columns
                elif any(t in ctype for t in ["NUMBER", "INT", "DECIMAL", "FLOAT"]):
                    df = df.filter(sql_expr(f"TRY_CAST({cname} AS NUMBER) IS NOT NULL"))

                # Cast date/timestamp columns
                elif any(t in ctype for t in ["DATE", "TIMESTAMP"]):
                    df = df.with_column(cname, col(cname).cast("timestamp"))

            # ---------------------------------------
            # 3. START_DATE handling
            # ---------------------------------------
            if 'START_DATE' in cols_upper:
                start_col = cols[cols_upper.index('START_DATE')]

                df = df.filter(col(start_col).is_not_null())
                df = df.filter(col(start_col) <= now)
                df = df.with_column(start_col, col(start_col).cast("timestamp"))
            else:
                df = df.with_column("START_DATE", now)

            # ---------------------------------------
            # 4. END_DATE handling
            # ---------------------------------------
            if 'END_DATE' in cols_upper:
                end_col = cols[cols_upper.index('END_DATE')]

                df = df.filter(col(end_col).is_not_null())
                df = df.with_column(end_col, col(end_col).cast("timestamp"))

                # Ensure END_DATE >= START_DATE
                if 'START_DATE' in cols_upper:
                    start_col = cols[cols_upper.index('START_DATE')]
                    df = df.filter(col(end_col) >= col(start_col))
            else:
                df = df.with_column("END_DATE", max_end_date)

            # ---------------------------------------
            # 5. SAFE NULL FILTERING (only critical cols)
            # ---------------------------------------
            required_cols = ["START_DATE", "END_DATE"]
            not_null_filters = [col(c).is_not_null() for c in required_cols if c in cols]

            if not_null_filters:
                df = df.filter(reduce(operator.and_, not_null_filters))

            # ---------------------------------------
            # 6. ADD LOAD_DATE
            # ---------------------------------------
            df = df.with_column("LOAD_DATE", now)

            # ---------------------------------------
            # 7. DEDUPLICATE
            # ---------------------------------------
            df = df.distinct()

            # ---------------------------------------
            # 8. REORDER COLUMNS (DELETE_FLAG LAST)
            # ---------------------------------------
            cols = df.columns
            cols_upper = [c.upper() for c in cols]

            if "DELETE_FLAG" in cols_upper:
                idx = cols_upper.index("DELETE_FLAG")
                delete_col = cols[idx]

                reordered_cols = [c for c in cols if c != delete_col] + [delete_col]
                df = df.select(reordered_cols)

            # ---------------------------------------
            # 9. SAVE TO SILVER
            # ---------------------------------------
            df.write.mode("overwrite").save_as_table(silver_full)

            successes.append(silver_full)

        except Exception as e:
            failures.append(f"{bronze_table}: {str(e)}")

    if failures:
        return f"Partial success. Created {len(successes)} tables. Failures: {failures}"
    else:
        return f"Success. Created {len(successes)} tables: {', '.join(successes)}"
$$;

In [ ]:
CALL PROC_CREATE_SILVER(
  'DEV',
  'DATASCIENCE',
  ARRAY_CONSTRUCT(
    OBJECT_CONSTRUCT('bronze_table','BRONZE_CUSTOMERS_DELTA','silver_table','SILVER_CUSTOMERS_DELTA'),
    OBJECT_CONSTRUCT('bronze_table','BRONZE_ORDERS_DELTA','silver_table','SILVER_ORDERS_DELTA'), OBJECT_CONSTRUCT('bronze_table','BRONZE_PRODUCTS_DELTA','silver_table','SILVER_PRODUCTS_DELTA')
  )
);


In [ ]:
select * from SILVER_PRODUCTS_DELTA ORDER by PRODUCT_ID